# 23 高级Transformer主题 - RoPE、Flash Attention、KV Cache、GQA、SwiGLU

本教程介绍现代大语言模型（LLaMA、GPT等）使用的**高级Transformer优化技术**。这些技术让Transformer能处理更长序列、更高效地推理。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("导入库成功！")

## 一、技术总览

| 技术 | 解决的问题 | 首次出现 |
|------|-----------|----------|
| **RoPE** | 传统位置编码无法外推到长序列 | RoFormer, 2021 |
| **Flash Attention** | 注意力计算的内存瓶颈 | Dao et al., 2022 |
| **KV Cache** | 推理时重复计算Key/Value | 通用优化 |
| **GQA** | 多头注意力的KV内存开销 | PaLM, LLaMA 3 |
| **SwiGLU** | ReLU表达能力不足 | PaLM, LLaMA |

## 二、RoPE（旋转位置编码）

### 传统位置编码的问题

原始Transformer用正弦/余弦函数生成位置编码，然后**加到词嵌入上**。
这种方式有个致命问题：**训练时没见过的位置，推理时编码完全不同**。

比如训练时最长序列是512，推理时遇到第513个位置，编码模式无法泛化。

### RoPE的核心思想

**不直接加位置信息，而是通过旋转变换注入位置**。

对Query和Key向量做旋转：

$$\begin{bmatrix} q_{2i}' \\ q_{2i+1}' \end{bmatrix} = \begin{bmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{bmatrix} \begin{bmatrix} q_{2i} \\ q_{2i+1} \end{bmatrix}$$

其中：
- $m$ 是位置（0, 1, 2, ...）
- $\theta_i = 10000^{-2i/d}$ 是频率
- 每对维度 $(2i, 2i+1)$ 做不同角度的旋转

### 为什么旋转编码更好？

旋转后，两个位置的Query和Key的点积只依赖于它们的**相对距离**：

$$(R_m q)^T (R_n k) = q^T R_{n-m} k$$

这意味着模型学到的是相对位置关系，可以**自然外推到任意长度**。

In [ ]:
# RoPE实现
print("=" * 70)
print("RoPE (Rotary Positional Embedding)")
print("=" * 70)
print()

def apply_rope(q, k, positions=None):
    """
    对Q和K应用旋转位置编码
    q: (seq_len, d_model)
    k: (seq_len, d_model)
    positions: (seq_len,) 如果为None，则用 0, 1, 2, ...
    """
    seq_len, d_model = q.shape
    if positions is None:
        positions = torch.arange(seq_len)
    
    # 计算旋转角度
    # θ_i = 10000^(-2i/d)
    half_d = d_model // 2
    freqs = 1.0 / (10000 ** (torch.arange(0, half_d * 2, 2).float() / d_model))
    
    # 对每个位置计算角度
    # t: (seq_len, half_d)
    t = positions.float().unsqueeze(1) * freqs.unsqueeze(0)
    
    # 构建旋转矩阵
    cos_t = torch.cos(t)  # (seq_len, half_d)
    sin_t = torch.sin(t)  # (seq_len, half_d)
    
    # 将Q和K分为前后两半
    q1, q2 = q[..., :half_d], q[..., half_d:]
    k1, k2 = k[..., :half_d], k[..., half_d:]
    
    # 应用旋转: [q1, q2] → [q1*cos - q2*sin, q1*sin + q2*cos]
    q_rotated = torch.cat([q1 * cos_t - q2 * sin_t, q1 * sin_t + q2 * cos_t], dim=-1)
    k_rotated = torch.cat([k1 * cos_t - k2 * sin_t, k1 * sin_t + k2 * cos_t], dim=-1)
    
    return q_rotated, k_rotated

# 数值示例
d_model = 8
seq_len = 6
q = torch.randn(seq_len, d_model)
k = torch.randn(seq_len, d_model)

print(f"原始Q形状: {q.shape}")
print(f"原始K形状: {k.shape}")

q_rope, k_rope = apply_rope(q, k)
print(f"RoPE后Q形状: {q_rope.shape}")
print(f"RoPE后K形状: {k_rope.shape}")
print()

# 展示旋转效果
print(f"旋转角度 (前4个位置, 前2对维度):")
half_d = d_model // 2
freqs = 1.0 / (10000 ** (torch.arange(0, half_d * 2, 2).float() / d_model))
print(f"  频率 θ_i: {freqs.numpy()}")
print()

for pos in range(min(4, seq_len)):
    angles = pos * freqs
    print(f"  位置 {pos}: 角度 = {angles.numpy()}, cos = {torch.cos(angles).numpy()}, sin = {torch.sin(angles).numpy()}")

# 验证相对位置编码
print("\n相对位置编码验证:")
print("  如果RoPE正确，q_i · k_j 应该只依赖于 (j - i)")
print()

# 计算旋转前后的点积
scores_before = q @ k.T
scores_after = q_rope @ k_rope.T

print(f"旋转前的点积矩阵:")
for i in range(seq_len):
    print(f"  q{i}: ", end="")
    for j in range(seq_len):
        print(f"{scores_before[i,j]:>8.3f}", end="")
    print()

print(f"\n旋转后的点积矩阵:")
for i in range(seq_len):
    print(f"  q{i}: ", end="")
    for j in range(seq_len):
        print(f"{scores_after[i,j]:>8.3f}", end="")
    print()

In [ ]:
# 可视化RoPE的旋转效果
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. 旋转角度随位置变化
d_model_viz = 64
half_d_viz = d_model_viz // 2
freqs_viz = 1.0 / (10000 ** (torch.arange(0, half_d_viz * 2, 2).float() / d_model_viz))

positions = torch.arange(0, 100)
angles = positions.unsqueeze(1) * freqs_viz.unsqueeze(0)

im = axes[0, 0].imshow(angles.numpy().T, cmap='viridis', aspect='auto')
axes[0, 0].set_xlabel('Position')
axes[0, 0].set_ylabel('Dimension Pair Index')
axes[0, 0].set_title('RoPE Rotation Angle')
fig.colorbar(im, ax=axes[0, 0], label='角度 (弧度)')

# 2. 某个维度对上的旋转轨迹
dim_idx = 0
q_sample = torch.tensor([1.0, 0.0])  # 初始向量
trajectory = []
for pos in range(50):
    angle = (pos * freqs_viz[dim_idx]).item()
    cos_a, sin_a = math.cos(angle), math.sin(angle)
    rotated = torch.tensor([
        q_sample[0] * cos_a - q_sample[1] * sin_a,
        q_sample[0] * sin_a + q_sample[1] * cos_a
    ])
    trajectory.append(rotated.numpy())

trajectory = np.array(trajectory)
axes[0, 1].plot(trajectory[:, 0], trajectory[:, 1], 'b-o', markersize=3, alpha=0.5)
axes[0, 1].scatter([1], [0], color='red', s=100, zorder=5, label='初始向量 (1,0)')
axes[0, 1].set_xlabel('Dimension 2i')
axes[0, 1].set_ylabel(f'Dimension 2i+1')
axes[0, 1].set_title(f'Rotation Transform on Dimension Pair {dim_idx+1}')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_aspect('equal')

# 3. 不同维度的频率衰减
axes[1, 0].semilogy(range(len(freqs_viz)), freqs_viz.numpy(), 'b-o', markersize=3)
axes[1, 0].set_xlabel('Dimension Pair Index i')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('RoPE频率衰减 (对数坐标)\nθ_i = 10000^(-2i/d)')
axes[1, 0].grid(True, alpha=0.3)

# 4. 点积随相对位置的变化
# 创建两个固定向量，在不同相对位置下的点积
d_model_test = 16
q_fixed = torch.randn(d_model_test)
k_fixed = torch.randn(d_model_test)

max_pos = 40
relative_positions = np.arange(-max_pos, max_pos+1)
dot_products = []

for rel_pos in relative_positions:
    q_test = q_fixed.clone()
    k_test = k_fixed.clone()
    
    # 应用RoPE
    q_test_2d = q_test.unsqueeze(0)
    k_test_2d = k_test.unsqueeze(0)
    
    q_pos, _ = apply_rope(q_test_2d, k_test_2d, torch.tensor([0]))
    _, k_pos = apply_rope(q_test_2d, k_test_2d, torch.tensor([abs(rel_pos)]))
    
    if rel_pos >= 0:
        dot = (q_pos @ k_pos.T).item()
    else:
        dot = (k_pos @ q_pos.T).item()
    dot_products.append(dot)

axes[1, 1].plot(relative_positions, dot_products, 'g-', linewidth=2)
axes[1, 1].axvline(x=0, color='r', linestyle='--', alpha=0.5)
axes[1, 1].set_xlabel('Relative Position (j - i)')
axes[1, 1].set_ylabel('Q.K Dot Product after RoPE')
axes[1, 1].set_title('点积随相对位置的变化\n(体现相对位置编码特性)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nRoPE的核心优势：")
print("  1. 相对位置编码：注意力只依赖于词之间的相对距离")
print("  2. 可外推：训练时没见过的长序列也能处理")
print("  3. 被LLaMA、GPT-Neo等广泛使用")

## 三、Flash Attention

### 标准注意力的瓶颈

标准注意力的计算：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

问题在于需要**显式构建 $QK^T$ 矩阵**，其大小为 $O(n^2)$：
- 序列长度 $n = 4096$ 时，$QK^T$ 矩阵占 $4096^2 \times 4$ 字节 ≈ 64MB
- $n = 32768$ 时，需要约 4GB！

这造成了**显存瓶颈**（GPU的HBM比SRAM慢得多）。

### Flash Attention的思想

Flash Attention通过**分块计算（Tiling）**和**重计算（Recomputation）**：
1. 不显式构建完整的 $QK^T$ 矩阵
2. 将Q、K、V分成小块，逐块计算
3. 在线更新softmax的归一化因子

核心公式（在线softmax）：

$$m_i = \max(m_{i-1}, \text{block}_i)$$
$$l_i = l_{i-1} \cdot e^{m_{i-1} - m_i} + e^{\text{block}_i - m_i}$$

最终：
$$\text{softmax}(x_j) = \frac{e^{x_j - m}}{l}$$

### 效果

- 内存复杂度：从 $O(n^2)$ 降到 $O(n)$
- 速度提升：2-4倍（取决于序列长度）
- 支持更长序列：理论上可以处理任意长度的序列（受限于时间而非显存）

In [ ]:
# Flash Attention概念演示
print("=" * 70)
print("Flash Attention 概念演示")
print("=" * 70)
print()

def standard_attention(Q, K, V):
    """标准注意力（需要完整QK^T矩阵）"""
    d_k = Q.size(-1)
    scores = Q @ K.T / math.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

def flash_attention_concept(Q, K, V, block_size=2):
    """
    Flash Attention概念版本（简化，非最优实现）
    通过分块计算，避免显式构建完整QK^T矩阵
    """
    seq_len = Q.size(0)
    d_k = Q.size(-1)
    d_v = V.size(-1)
    
    # 输出和统计量
    output = torch.zeros(seq_len, d_v)
    row_max = torch.full((seq_len,), float('-inf'))
    row_sum = torch.zeros(seq_len)
    all_weights = torch.zeros(seq_len, seq_len)  # 仅为了对比，实际不需要
    
    # 分块处理K和V
    for j in range(0, seq_len, block_size):
        K_block = K[j:j+block_size]
        V_block = V[j:j+block_size]
        
        # 计算当前块对每个Q的贡献
        for i in range(0, seq_len, block_size):
            Q_block = Q[i:i+block_size]
            
            # 当前块的注意力分数
            block_scores = Q_block @ K_block.T / math.sqrt(d_k)
            
            # 记录分数（为了可视化）
            all_weights[i:i+block_size, j:j+block_size] = block_scores
            
            # 在线softmax更新
            block_max = block_scores.max(dim=-1, keepdim=True).values
            block_exp = torch.exp(block_scores - block_max)
            block_sum = block_exp.sum(dim=-1)
            block_output = block_exp @ V_block
            
            # 更新全局统计量
            new_max = torch.maximum(row_max[i:i+block_size], block_max.squeeze())
            alpha = torch.exp(row_max[i:i+block_size] - new_max)
            beta = torch.exp(block_max.squeeze() - new_max)
            
            new_sum = row_sum[i:i+block_size] * alpha + block_sum * beta
            new_output = (output[i:i+block_size] * row_sum[i:i+block_size].unsqueeze(1) * alpha.unsqueeze(1) + 
                         block_output * beta.unsqueeze(1))
            
            output[i:i+block_size] = new_output
            row_max[i:i+block_size] = new_max
            row_sum[i:i+block_size] = new_sum
    
    # 归一化
    output = output / row_sum.unsqueeze(1)
    
    # 计算完整权重矩阵用于对比
    final_weights = F.softmax(all_weights / math.sqrt(d_k), dim=-1)
    
    return output, final_weights

# 对比标准注意力和Flash Attention
torch.manual_seed(42)
seq_len = 8
d_k = 4

Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

print(f"序列长度: {seq_len}, 维度: {d_k}")
print()

# 标准注意力
output_std, weights_std = standard_attention(Q, K, V)

# Flash Attention概念
output_flash, weights_flash = flash_attention_concept(Q, K, V, block_size=2)

print("对比结果:")
print(f"  输出差异: {(output_std - output_flash).abs().max().item():.6f}")
print(f"  权重差异: {(weights_std - weights_flash).abs().max().item():.6f}")
print()

# 内存使用对比
large_seq = 1024
Q_large = torch.randn(large_seq, d_k)
K_large = torch.randn(large_seq, d_k)

print(f"大序列内存使用 (seq_len={large_seq}):")
print(f"  QK^T矩阵大小: {large_seq} × {large_seq} × 4字节 = {large_seq*large_seq*4/1024:.1f} KB")
print(f"  Flash Attention块大小 (block=32): {32*32*4/1024:.1f} KB 每次")
print(f"  内存节省: 约 {large_seq*large_seq / (32*32):.0f} 倍")

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im1 = axes[0].imshow(weights_std.detach().numpy(), cmap='Blues')
axes[0].set_title('Standard Attention Weights')
fig.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(weights_flash.detach().numpy(), cmap='Blues')
axes[1].set_title('Flash Attention Weights')
fig.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow((weights_std - weights_flash).abs().detach().numpy(), cmap='Reds')
axes[2].set_title('Difference (should be near 0)')
fig.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

## 四、KV Cache（推理优化）

### 问题

在**自回归推理**（逐步生成token）时，每生成一个新token，都需要重新计算整个序列的注意力。

比如生成到第100个token时：
- 标准做法：重新计算100个token的Q, K, V
- 但实际上：前99个token的K, V已经计算过了，不需要重算！

### KV Cache

**缓存之前计算的K和V，每步只计算新token的K和V。**

```
不用KV Cache:                    使用KV Cache:
Step 1: Attn(Q₁, K₁, V₁)        Step 1: Attn(Q₁, K₁, V₁) → Cache K₁,V₁
Step 2: Attn(Q₂, K₁:₂, V₁:₂)    Step 2: Attn(Q₂, [K₁, K₂], [V₁, V₂]) → Cache K₁:₂,V₁:₂
Step 3: Attn(Q₃, K₁:₃, V₁:₃)    Step 3: Attn(Q₃, [K₁:₂, K₃], [V₁:₂, V₃]) → 只算K₃,V₃
```

这大幅减少了推理时的计算量，尤其是长序列生成时。

In [ ]:
# KV Cache 演示
print("=" * 70)
print("KV Cache - 推理加速")
print("=" * 70)
print()

class AttentionWithKVCache:
    def __init__(self, d_model):
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.d_model = d_model
        self.kv_cache = None
    
    def forward_with_cache(self, x, use_cache=False):
        """
        x: (1, 1, d_model) 单个token
        """
        q = self.W_Q(x)  # (1, 1, d_model)
        k = self.W_K(x)  # (1, 1, d_model)
        v = self.W_V(x)  # (1, 1, d_model)
        
        if use_cache and self.kv_cache is not None:
            # 拼接缓存的K,V和新的K,V
            k_cached, v_cached = self.kv_cache
            k = torch.cat([k_cached, k], dim=1)  # (1, seq_len+1, d_model)
            v = torch.cat([v_cached, v], dim=1)
        
        # 更新缓存
        if use_cache:
            if self.kv_cache is None:
                self.kv_cache = (k, v)
            else:
                k_cached, v_cached = self.kv_cache
                self.kv_cache = (torch.cat([k_cached, k[:, -1:]], dim=1),
                                 torch.cat([v_cached, v[:, -1:]], dim=1))
        
        # 注意力计算
        d_k = self.d_model
        scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
        weights = F.softmax(scores, dim=-1)
        output = weights @ v
        return self.W_O(output)
    
    def reset_cache(self):
        self.kv_cache = None

# 演示推理过程
torch.manual_seed(42)
d_model = 16
model = AttentionWithKVCache(d_model)

print("推理过程对比（生成5个token）:")
print()
print("不用KV Cache:")
for step in range(1, 6):
    # 不用cache时，需要处理整个序列
    seq_len = step
    x_full = torch.randn(1, seq_len, d_model)
    q_full = model.W_Q(x_full)
    k_full = model.W_K(x_full)
    v_full = model.W_V(x_full)
    # 计算量: seq_len × seq_len
    flops = seq_len * seq_len
    print(f"  Step {step}: 处理序列长度 {seq_len}, 注意力计算量 O({seq_len}×{seq_len}) = O({flops})")

print()
print("使用KV Cache:")
model.reset_cache()
total_flops_cache = 0
for step in range(1, 6):
    # 使用cache时，只需计算新token
    x_new = torch.randn(1, 1, d_model)
    output = model.forward_with_cache(x_new, use_cache=True)
    # 计算量: seq_len × 1 (Q只和新token算)
    flops = step * 1
    total_flops_cache += flops
    print(f"  Step {step}: 计算新token Q, K, V. 注意力 O({step}×1) = O({flops})")

print()
total_flops_no_cache = sum(i*i for i in range(1, 6))
print(f"总计算量对比:")
print(f"  不用KV Cache: {total_flops_no_cache}")
print(f"  使用KV Cache: {total_flops_cache}")
print(f"  加速比: {total_flops_no_cache/total_flops_cache:.1f} 倍")
print(f"\n序列越长，KV Cache的加速效果越明显！")

In [ ]:
# KV Cache的内存开销
print("\n=== KV Cache内存开销 ===")
print()

def kv_cache_memory(n_layers, n_heads, seq_len, head_dim, batch_size=1, dtype_bytes=4):
    """计算KV Cache占用的内存"""
    # 每层每头的K和V: (batch, seq_len, head_dim)
    per_head = batch_size * seq_len * head_dim * dtype_bytes
    total = n_layers * n_heads * 2 * per_head  # ×2 因为K和V
    return total

# LLaMA-3-8B 的配置
n_layers = 32
n_heads = 32
head_dim = 128

print(f"LLaMA-3-8B配置: {n_layers}层, {n_heads}头, head_dim={head_dim}")
print()

for seq_len in [512, 2048, 8192, 32768]:
    mem = kv_cache_memory(n_layers, n_heads, seq_len, head_dim)
    mem_mb = mem / (1024*1024)
    print(f"  序列长度 {seq_len:>6}: KV Cache = {mem_mb:>10.1f} MB ({mem_mb/1024:.1f} GB)")

print("\n注意: KV Cache可以占用大量显存，尤其在长序列生成时")
print("这也是为什么GQA（下面介绍）重要的原因——它减少了K/V的头数")

## 五、GQA（分组查询注意力）

### 问题：MHA的KV内存开销

标准多头注意力（MHA）中，Q、K、V各有 `n_heads` 个。KV Cache占用与头数成正比。

### GQA的思路

**让多个Query头共享同一组Key和Value头。**

```
MHA:  Q1 Q2 Q3 Q4 Q5 Q6 Q7 Q8    (8个Q头)
      K1 K2 K3 K4 K5 K6 K7 K8    (8个K头)
      V1 V2 V3 V4 V5 V6 V7 V8    (8个V头)

GQA:  Q1 Q2 Q3 Q4 Q5 Q6 Q7 Q8    (8个Q头)
      K1 K1 K2 K2 K3 K3 K4 K4    (4个K头, 每2个Q共享1个K)
      V1 V1 V2 V2 V3 V3 V4 V4    (4个V头)

MQA:  Q1 Q2 Q3 Q4 Q5 Q6 Q7 Q8    (8个Q头)
      K  K  K  K  K  K  K  K     (1个K头, 全部共享)
      V  V  V  V  V  V  V  V     (1个V头)
```

### 效果

| 方式 | K/V头数 | KV Cache | 质量 | 速度 |
|------|---------|----------|------|------|
| MHA | n_heads | 100% | 基准 | 基准 |
| GQA | n_heads/k | 1/k | 接近MHA | 快k倍 |
| MQA | 1 | 1/n_heads | 略低 | 最快 |

In [ ]:
# GQA 实现
print("=" * 70)
print("GQA (Grouped Query Attention)")
print("=" * 70)
print()

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups = n_heads // n_kv_heads  # 每组多少个头
        self.head_dim = d_model // n_heads
        
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.W_V = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        # 投影Q, K, V
        Q = self.W_Q(x).view(batch_size, seq_len, self.n_heads, self.head_dim)
        K = self.W_K(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim)
        V = self.W_V(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim)
        
        # 扩展K和V以匹配Q的头数
        # 每组的n_groups个Q头共享同一个K/V头
        K = K.repeat_interleave(self.n_groups, dim=2)  # (batch, seq, n_heads, head_dim)
        V = V.repeat_interleave(self.n_groups, dim=2)
        
        # Scaled dot-product attention
        Q = Q.transpose(1, 2)  # (batch, n_heads, seq, head_dim)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        output = (attn_weights @ V).transpose(1, 2).contiguous()
        output = output.view(batch_size, seq_len, -1)
        
        return self.W_O(output), attn_weights

# 对比不同注意力方式
torch.manual_seed(42)
d_model = 64
n_heads = 8
seq_len = 16

print(f"配置: d_model={d_model}, n_heads={n_heads}, seq_len={seq_len}")
print()

configs = [
    ('MHA (标准多头)', n_heads, n_heads),
    ('GQA (分组查询, 4组)', n_heads, n_heads // 2),
    ('GQA (分组查询, 2组)', n_heads, n_heads // 4),
    ('MQA (多查询)', n_heads, 1),
]

x = torch.randn(1, seq_len, d_model)

print(f"{'类型':<30} {'K/V头数':>8} {'KV Cache':>10} {'参数量(K/V)':>12}")
print("-" * 65)

for name, n_h, n_kv_h in configs:
    gqa = GroupedQueryAttention(d_model, n_h, n_kv_h)
    output, attn = gqa(x)
    
    kv_params = sum(p.numel() for p in [gqa.W_K, gqa.W_V])
    kv_cache_ratio = n_kv_h / n_heads * 100
    
    print(f"{name:<30} {n_kv_h:>8} {kv_cache_ratio:>9.0f}% {kv_params:>12}")

print()
print("结论:")
print("  - GQA在KV Cache和参数数量上比MHA节省很多")
print("  - MQA最省，但质量可能略低")
print("  - LLaMA 3 使用GQA (n_heads=32, n_kv_heads=8, 比例4:1)")

## 六、SwiGLU激活函数

### 从ReLU到SwiGLU

原始Transformer的FFN用ReLU：
$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

现代LLM使用SwiGLU：
$$\text{SwiGLU}(x) = \text{Swish}(xW_1) \odot (xW_2)$$
$$\text{FFN}_{\text{SwiGLU}}(x) = \text{SwiGLU}(x)W_3$$

其中 $\text{Swish}(x) = x \cdot \sigma(\beta x)$，通常 $\beta = 1$。

### 为什么SwiGLU更好？

1. **门控机制**：类似LSTM的门控，让模型学会控制信息流
2. **更平滑的梯度**：Swish在零点附近比ReLU平滑，梯度传播更好
3. **更强的表达能力**：两路输入（一个经过Swish，一个直接）允许更复杂的非线性

### 参数量变化

- ReLU FFN: 2个矩阵 ($W_1, W_2$)
- SwiGLU FFN: 3个矩阵 ($W_1, W_2, W_3$)

虽然参数量增加了50%，但模型质量提升明显。LLaMA通过缩小FFN中间层维度来平衡参数量。

In [ ]:
# SwiGLU 实现和对比
print("=" * 70)
print("SwiGLU 激活函数")
print("=" * 70)
print()

class SwiGLU(nn.Module):
    """
    SwiGLU = Swish(xW1) ⊗ (xW2)
    """
    def forward(self, x1, x2):
        return F.silu(x1) * x2

class FFN_ReLU(nn.Module):
    """原始Transformer的FFN"""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

class FFN_SwiGLU(nn.Module):
    """SwiGLU版本的FFN"""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)  # gate
        self.w2 = nn.Linear(d_model, d_ff, bias=False)  # value
        self.w3 = nn.Linear(d_ff, d_model, bias=False)  # output
        self.swiglu = SwiGLU()
    
    def forward(self, x):
        return self.w3(self.swiglu(self.w1(x), self.w2(x)))

class FFN_GELU(nn.Module):
    """BERT/GPT-2使用的GELU FFN"""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()
    
    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

# 对比
torch.manual_seed(42)
d_model = 64
d_ff_relu = 128
d_ff_swiglu = 86  # SwiGLU通常用更小的d_ff来平衡参数量 (128*2/3 ≈ 86)

x = torch.randn(1, 10, d_model)

print(f"配置: d_model={d_model}")
print(f"  ReLU FFN: d_ff={d_ff_relu}")
print(f"  SwiGLU FFN: d_ff={d_ff_swiglu}")
print()

ffns = [
    ('ReLU FFN', FFN_ReLU(d_model, d_ff_relu)),
    ('GELU FFN', FFN_GELU(d_model, d_ff_relu)),
    ('SwiGLU FFN', FFN_SwiGLU(d_model, d_ff_swiglu)),
]

print(f"{'类型':<15} {'参数量':>8} {'参数量比例':>12} {'输出范围':>18}")
print("-" * 58)

for name, ffn in ffns:
    n_params = sum(p.numel() for p in ffn.parameters())
    with torch.no_grad():
        out = ffn(x)
    out_range = f"[{out.min().item():.3f}, {out.max().item():.3f}]"
    ratio = n_params / sum(p.numel() for p in FFN_ReLU(d_model, d_ff_relu).parameters()) * 100
    
    print(f"{name:<15} {n_params:>8} {ratio:>11.0f}% {out_range:>18}")

# 可视化激活函数
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

x_1d = torch.linspace(-3, 3, 200)

# ReLU
axes[0].plot(x_1d.numpy(), F.relu(x_1d).numpy(), 'b-', linewidth=2)
axes[0].set_title('ReLU\nmax(0, x)')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axvline(x=0, color='k', linewidth=0.5)

# GELU
axes[1].plot(x_1d.numpy(), F.gelu(x_1d).numpy(), 'g-', linewidth=2)
axes[1].set_title('GELU\nGaussian Error Linear Unit')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].axvline(x=0, color='k', linewidth=0.5)

# Swish (SwiGLU中的激活)
axes[2].plot(x_1d.numpy(), F.silu(x_1d).numpy(), 'r-', linewidth=2)
axes[2].set_title('Swish/SiLU\nx · sigmoid(x)')
axes[2].grid(True, alpha=0.3)
axes[2].axhline(y=0, color='k', linewidth=0.5)
axes[2].axvline(x=0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

print("\nSwiGLU优势：")
print("  1. Swish在0点附近可导，梯度传播比ReLU好")
print("  2. 门控机制(xW1⊗xW2)让模型能控制信息流")
print("  3. LLaMA/PaLM等现代LLM都在用SwiGLU")

## 七、现代LLM的架构汇总

### LLaMA-3 架构

LLaMA-3 集成了本教程介绍的所有技术：

| 组件 | 选择 |
|------|------|
| 位置编码 | **RoPE** |
| 注意力 | **GQA** (32个Q头, 8个KV头) |
| FFN激活 | **SwiGLU** |
| 归一化 | **RMSNorm** (在Pre-Norm之前) |
| 注意力优化 | **Flash Attention 2** |
| 推理优化 | **KV Cache** |

### 从原始Transformer到现代LLM的演变

| 特性 | Transformer(2017) | LLaMA-3(2024) |
|------|-------------------|---------------|
| 位置编码 | 正弦/余弦加法 | RoPE旋转 |
| 注意力 | MHA | GQA + Flash Attention |
| FFN | ReLU | SwiGLU |
| 归一化 | Post-LayerNorm | Pre-RMSNorm |
| 推理 | 无优化 | KV Cache |
| 最大上下文 | 512 | 128K+ |

In [ ]:
# 现代LLM架构对比
print("=" * 70)
print("现代LLM架构对比")
print("=" * 70)
print()

architectures = {
    'Transformer (2017)': {
        '位置编码': '正弦加法',
        '注意力': 'MHA',
        'FFN': 'ReLU',
        '归一化': 'Post-LayerNorm',
        '最大上下文': '512',
    },
    'BERT (2018)': {
        '位置编码': '学习加法',
        '注意力': 'MHA',
        'FFN': 'GELU',
        '归一化': 'Post-LayerNorm',
        '最大上下文': '512',
    },
    'GPT-2 (2019)': {
        '位置编码': '学习加法',
        '注意力': 'MHA',
        'FFN': 'GELU',
        '归一化': 'Pre-LayerNorm',
        '最大上下文': '1024',
    },
    'LLaMA (2023)': {
        '位置编码': 'RoPE',
        '注意力': 'GQA',
        'FFN': 'SwiGLU',
        '归一化': 'Pre-RMSNorm',
        '最大上下文': '4096',
    },
    'LLaMA-3 (2024)': {
        '位置编码': 'RoPE (插值缩放)',
        '注意力': 'GQA + Flash Attn 2',
        'FFN': 'SwiGLU',
        '归一化': 'Pre-RMSNorm',
        '最大上下文': '128K',
    },
}

# 打印对比表
headers = ['特性'] + list(architectures.keys())
features = ['位置编码', '注意力', 'FFN', '归一化', '最大上下文']

# 打印表头
max_name_len = max(len(h) for h in headers)
col_width = 22

print(f"{'特性':<{max_name_len}}", end="")
for h in headers[1:]:
    print(f"{h:>{col_width}}", end="")
print()
print("-" * (max_name_len + col_width * len(headers[1:])))

for feat in features:
    print(f"{feat:<{max_name_len}}", end="")
    for model in headers[1:]:
        val = architectures[model][feat]
        print(f"{val:>{col_width}}", end="")
    print()

print()
print("技术演变趋势:")
print("  1. 位置编码: 固定加法 → 学习加法 → 旋转编码(RoPE)")
print("  2. 注意力:   MHA → GQA (减少KV开销)")
print("  3. FFN:      ReLU → GELU → SwiGLU (门控增强表达力)")
print("  4. 归一化:   Post → Pre + LayerNorm → RMSNorm")
print("  5. 上下文:   512 → 128K+ (256倍增长)")

## 八、集成所有技术的现代Transformer

让我们把所有技术集成到一个完整的现代Transformer模型中。

In [ ]:
# 现代Transformer集成
print("=" * 70)
print("现代Transformer (集成RoPE + GQA + SwiGLU)")
print("=" * 70)
print()

class ModernTransformerBlock(nn.Module):
    """
    现代Transformer块：
    - RoPE位置编码
    - GQA注意力
    - SwiGLU FFN
    - Pre-RMSNorm
    """
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.n_groups = n_heads // n_kv_heads
        
        # RMSNorm
        self.attn_norm = nn.RMSNorm(d_model)
        self.ffn_norm = nn.RMSNorm(d_model)
        
        # GQA注意力
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.W_V = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        
        # SwiGLU FFN
        self.w1 = nn.Linear(d_model, d_ff, bias=False)  # gate
        self.w2 = nn.Linear(d_model, d_ff, bias=False)  # value
        self.w3 = nn.Linear(d_ff, d_model, bias=False)  # output
        
        self.dropout = nn.Dropout(dropout)
    
    def apply_rope(self, x, positions):
        """应用RoPE到x (分成head维度)"""
        # x: (batch, seq_len, n_heads, head_dim)
        batch, seq_len, n_heads, head_dim = x.shape
        half_d = head_dim // 2
        
        freqs = 1.0 / (10000 ** (torch.arange(0, half_d * 2, 2, device=x.device).float() / head_dim))
        t = positions.float().unsqueeze(1) * freqs.unsqueeze(0)
        cos_t = torch.cos(t)
        sin_t = torch.sin(t)
        
        x1, x2 = x[..., :half_d], x[..., half_d:]
        x_rotated = torch.cat([x1 * cos_t - x2 * sin_t, x1 * sin_t + x2 * cos_t], dim=-1)
        return x_rotated
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        positions = torch.arange(seq_len, device=x.device)
        
        # === 自注意力 (带RoPE + GQA) ===
        x_norm = self.attn_norm(x)
        
        Q = self.W_Q(x_norm).view(batch_size, seq_len, self.n_heads, self.head_dim)
        K = self.W_K(x_norm).view(batch_size, seq_len, self.n_kv_heads, self.head_dim)
        V = self.W_V(x_norm).view(batch_size, seq_len, self.n_kv_heads, self.head_dim)
        
        # 应用RoPE
        Q = self.apply_rope(Q, positions)
        K = self.apply_rope(K, positions)
        
        # GQA: 扩展K和V
        K = K.repeat_interleave(self.n_groups, dim=2)
        V = V.repeat_interleave(self.n_groups, dim=2)
        
        # Attention
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        attn_out = (attn_weights @ V).transpose(1, 2).contiguous()
        attn_out = attn_out.view(batch_size, seq_len, -1)
        
        x = x + self.dropout(self.W_O(attn_out))
        
        # === SwiGLU FFN ===
        x_norm = self.ffn_norm(x)
        gate = F.silu(self.w1(x_norm))
        value = self.w2(x_norm)
        ffn_out = self.w3(gate * value)
        
        x = x + self.dropout(ffn_out)
        return x

# 创建模型
torch.manual_seed(42)
d_model = 128
n_heads = 8
n_kv_heads = 2  # GQA: 8个Q头共享2个KV头 (每组4个)
d_ff = 170      # SwiGLU: 比标准FFN小一些以平衡参数
n_layers = 4

layers = [ModernTransformerBlock(d_model, n_heads, n_kv_heads, d_ff) for _ in range(n_layers)]
model_modern = nn.Sequential(*layers)

print(f"现代Transformer块配置:")
print(f"  d_model = {d_model}")
print(f"  n_heads (Q) = {n_heads}")
print(f"  n_kv_heads (K/V) = {n_kv_heads} (GQA比例 {n_heads//n_kv_heads}:1)")
print(f"  d_ff (SwiGLU) = {d_ff}")
print(f"  n_layers = {n_layers}")
print()
print(f"总参数量: {sum(p.numel() for p in model_modern.parameters()):,}")

# 测试前向传播
x = torch.randn(1, 16, d_model)
output = model_modern(x)

print(f"\n前向传播测试:")
print(f"  输入: {x.shape}")
print(f"  输出: {output.shape}")
print(f"  输出范围: [{output.min().item():.4f}, {output.max().item():.4f}]")
print(f"  输出均值: {output.mean().item():.4f}")
print(f"  输出标准差: {output.std().item():.4f}")

## 九、全系列教程总结

### 完整知识体系

恭喜！你已经完成了从神经网络基础到Transformer的完整学习路径：

| 教程 | 核心内容 |
|------|----------|
| **15** | 神经网络基础：感知机、激活函数、梯度下降 |
| **16** | 优化算法：SGD、Momentum、Adam |
| **17** | 反向传播：链式法则、梯度消失/爆炸 |
| **18** | RNN基础：前向传播、BPTT |
| **19** | LSTM/GRU：门控机制解决长距离依赖 |
| **20** | Seq2Seq+Attention：编码器-解码器架构 |
| **21** | Transformer架构：自注意力、多头、位置编码 |
| **22** | Transformer训练：从零训练完成实际任务 |
| **23** | 高级技术：RoPE、Flash Attention、KV Cache、GQA、SwiGLU |

### 你已经掌握了

1. **数学基础**：梯度下降、链式法则、注意力计算
2. **架构理解**：从RNN到Transformer的完整演变
3. **训练技巧**：优化器选择、学习率调度、梯度裁剪
4. **现代技术**：RoPE、GQA、SwiGLU等LLM核心技术

### 下一步建议

1. **阅读原始论文**：
   - "Attention Is All You Need" (Transformer)
   - "LLaMA" / "LLaMA-2" / "LLaMA-3" 技术报告

2. **动手实践**：
   - 用更大的数据集训练Transformer
   - 尝试实现Flash Attention
   - 微调一个预训练LLM

3. **深入学习**：
   - 研究RLHF（人类反馈强化学习）
   - 学习LoRA/QLoRA高效微调方法
   - 探索MoE（Mixture of Experts）架构

In [ ]:
# 最终总结
print("=" * 70)
print("🎉 恭喜完成Transformer全系列教程！")
print("=" * 70)
print()

print("你学到的知识体系：")
print()
print("  基础层 ──────────────────────────────────────────────")
print("    15 │ 神经网络基础 (感知机, 激活函数, 梯度下降)")
print("    16 │ 优化算法 (SGD, Momentum, Adam)")
print("    17 │ 反向传播 (链式法则, 梯度问题)")
print()
print("  序列模型层 ──────────────────────────────────────────")
print("    18 │ RNN基础 (前向传播, BPTT)")
print("    19 │ LSTM/GRU (门控机制)")
print("    20 │ Seq2Seq + Attention (编码器-解码器)")
print()
print("  Transformer层 ───────────────────────────────────────")
print("    21 │ Transformer架构 (自注意力, 多头, 位置编码)")
print("    22 │ Transformer训练 (从零训练)")
print("    23 │ 高级技术 (RoPE, GQA, SwiGLU, Flash Attention, KV Cache)")
print()
print("  ──────────────────────────────────────────────────────")
print()
print("继续加油！你已经有了理解和使用现代大语言模型的坚实基础。 🚀")